***type something cool later***

*notes:*

target is 5,174 (active) : 1,869 (churned).. class_weight='balanced'

In [1]:
import pandas as pd
import numpy as np
import math as mt
import warnings

data_contract = pd.read_csv('contract.csv')
data_internet = pd.read_csv('internet.csv')
data_personal = pd.read_csv('personal.csv')
data_phone = pd.read_csv('phone.csv')

#display(data_contract.head())
#display(data_internet.head())
#display(data_personal.head())
#display(data_phone.head())

In [2]:
    #!! converting date columns to datetime format
warnings.filterwarnings('ignore')
data_contract['BeginDate'] = pd.to_datetime(data_contract['BeginDate'])
data_contract['EndDate'] = pd.to_datetime(data_contract['EndDate'], errors='coerce')
warnings.filterwarnings('default')

    #!! filling NaT (EndDate = 'No') with 'current day' (data set extraction date 2020-02-01)
data_contract['EndDate'] = data_contract['EndDate'].fillna(pd.Timestamp('2020-02-01'))

    #!! calculating tenure in months
data_contract['tenure_month'] = (data_contract['EndDate'] - data_contract['BeginDate']).dt.days // 30

    #!! dropping BeginDate (no relationship with customer loyalty) & TotalCharges (can infer from tenure * monthly charges)
data_contract = data_contract.drop(['BeginDate', 'TotalCharges'], axis=1)

    #!! converting EndDate to boolean: 0 = active (2020-02-01), 1 = churned (other dates)
data_contract['EndDate'] = data_contract['EndDate'].apply(lambda x: 0 if x == pd.Timestamp('2020-02-01') else 1)

    #!! data_contract columns renamed
data_contract.rename(columns={'EndDate': 'churned', 'Type': 'contract_type', 'PaperlessBilling': 'paperless_billing',
                              'PaymentMethod': 'payment_method', 'MonthlyCharges': 'monthly_charges'}, inplace=True)

In [3]:
    #!! data_internet columns renamed
data_internet.rename(columns={'InternetService': 'internet_service', 'OnlineSecurity': 'online_security',
                              'OnlineBackup': 'online_backup', 'DeviceProtection': 'device_protection',
                              'TechSupport': 'tech_support', 'StreamingTV': 'streaming_tv',
                              'StreamingMovies': 'streaming_movies'}, inplace=True)

In [4]:
    #!! dropping gender column (no relationship with customer loyalty)
data_personal = data_personal.drop('gender', axis=1)

    #!! data_personal columns renamed
data_personal.rename(columns={'SeniorCitizen': 'senior_citizen', 'Partner': 'partner',
                              'Dependents': 'dependents'}, inplace=True)

In [5]:
    #!! data_phone columns renamed
data_phone.rename(columns={'MultipleLines': 'multiple_lines'}, inplace=True)

In [6]:
#data_contract.info()
#data_internet.info()
#data_personal.info()
#data_phone.info()

#display(data_contract.head())
#display(data_internet.head())
#display(data_personal.head())
#display(data_phone.head())

    #!! checking unique values
#for col in data_contract.select_dtypes(include=['object', 'int64']).columns: display(data_contract[col].value_counts())

    #!! checking unique values
#for col in data_internet.select_dtypes(include=['object']).columns: display(data_internet[col].value_counts())

    #!! checking unique values
#for col in data_personal.select_dtypes(include=['object', 'int64']).columns: display(data_personal[col].value_counts())

    #!! checking unique values
#for col in data_phone.select_dtypes(include=['object']).columns: display(data_phone[col].value_counts())

In [7]:
    #!! merging all dataframes on customerID
data = data_contract.merge(
    data_internet, on='customerID', how='left').merge(
        data_personal, on='customerID', how='left').merge(
            data_phone, on='customerID', how='left')

#data.info()
#display(data.head(20))
#display(data['multiple_lines'].value_counts())

    #!! filling 'No Service' for NaN values (customers without internet or phone service)
data.fillna('No Service', inplace=True)

    #!! creating binary columns for internet and phone service (0 = No Service, 1 = has service)
data['has_internet'] = data['internet_service'].apply(lambda x: 0 if x == 'No Service' else 1)
data['has_phone'] = data['multiple_lines'].apply(lambda x: 0 if x == 'No Service' else 1)

#data.info()
#display(data.head(20))
#display(data['multiple_lines'].value_counts())

**data cleaning finished, onto data split and preprocessing. (OHE, StandardScaler)**